# Bone Fracture Detection: CNN vs ScatNet, XAI, fracture boxes and a YOLO pipeline

*Visual Intelligence 2025/2026, University of Verona*

X-rays of **FracAtlas** (fractured / not fractured, with the radiologists' fracture boxes). We compare a CNN, a
ScatNet and a pretrained ResNet18 (all with **the same classifier**), explain them with **six XAI methods**
(Occlusion also from scratch), turn the explanations into **fracture boxes**, and build a **full pipeline**
after Linda (2025): best classifier + YOLOv8 + best XAI method -> a short report per X-ray.

All code is in `src/`; this notebook only sets parameters and calls it. Run it on Kaggle with `./run.sh push`.

| exam step | section |
|---|---|
| 2-3. dataset, train / test split | 1 |
| 4. CNN and ScatNet, same classifier | 2 |
| 5. k-fold CV: mean accuracy and F1 | 3 |
| 7. test set | 4 |
| choice of the best model | 5 |
| 6. filters | 6 |
| 8, 10, 11. six XAI methods, 2 images per class | 7 |
| 9. our Occlusion vs Captum | 8 |
| 10, 12. quality of the explanations: deletion test, boxes | 9, 10 |
| full pipeline (YOLO + classifier + XAI) | 11 |
| 12. discussion | 13 |

## Setup

In [ ]:
# Setup: on Kaggle ./run.sh push unpacks the code into /tmp/bfd; locally this repository is used
import sys
from pathlib import Path

ROOT = next(p for p in (Path("/tmp/bfd"), Path.cwd().parent, Path.cwd()) if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.bootstrap import setup
paths = setup(ROOT)   # installs kymatio, captum, ultralytics if missing; finds the dataset; makes the output folders

In [ ]:
# Settings: the only cell to edit
QUICK      = False     # True: 5-minute health check on a small subset (do it once before the full run)
TRAIN      = True      # False: reload the weights and results of a previous run from RESULTS_DIR
MODELS     = ["cnn", "scatnet", "resnet18"]
K_FOLDS    = 5
EPOCHS     = 20        # per fold and for the final training
BATCH_SIZE = 32
LR         = {"cnn": 1e-3, "scatnet": 1e-3, "resnet18": 1e-4}  # the pretrained ResNet18 is fine-tuned gently
SELECT_BY  = "f1"      # cross-validation metric that chooses the best model: "f1" or "accuracy"
DUP_THRESHOLD = 0.97   # thumbnail similarity above which two images are the same X-ray
XAI_PER_CLASS = 5      # test images explained per class (the figures show 2 per class)
XAI_PARAMS = dict(ig_steps=64, window=16, stride=8, lime_samples=1000)
BOX_THRESHOLD = 0.5    # a fracture box keeps the pixels above this share of the strongest evidence
YOLO_PARAMS = dict(model="yolov8s.pt", epochs=50, imgsz=640, batch=16)  # COCO-pretrained, as in FracAtlas
YOLO_CONF  = 0.25      # the detector reports a box only above this confidence
SEED = 42

MAX_PER_CLASS = None
if QUICK:
    MAX_PER_CLASS, K_FOLDS, EPOCHS, XAI_PER_CLASS = 60, 2, 1, 2
    XAI_PARAMS = dict(ig_steps=8, window=32, stride=16, lime_samples=50)
    YOLO_PARAMS = dict(model="yolov8n.yaml", epochs=1, imgsz=320, batch=8)   # small, from scratch, no download

import numpy as np, pandas as pd
from src import data, models, training, xai, occlusion_scratch, localize, detect, pipeline, plots, report, utils

DATA_DIR, RESULTS_DIR, FIG_DIR, CACHE_DIR = paths["data"], paths["results"], paths["figures"], paths["cache"]
utils.set_seed(SEED)
device = utils.get_device()
NUM_WORKERS = 3 if device.type == "cuda" else 0
print("device:", device, "| quick run" if QUICK else "| full run")

## 1. Dataset

**FracAtlas** (Abedeen et al., *Scientific Data* 2023): 4,083 X-rays of hands, legs, hips and shoulders from
three hospitals, 717 of them fractured. Two radiologists and an orthopaedist drew a **box around every
fracture**: section 10 uses these boxes to check whether the explanations point at the fracture.
Images are loaded in grey (1 channel), resized to 224x224 and kept in memory; the boxes are rescaled with them.
Only about 1 X-ray in 6 is fractured, so the loss weights each class by its inverse frequency (section 3).

In [ ]:
dataset = data.load_dataset(DATA_DIR, cache_dir=CACHE_DIR, max_per_class=MAX_PER_CLASS, seed=SEED)
class_names = dataset.class_names   # label 0 = fractured (the positive class), 1 = not fractured
print(f"{len(dataset)} X-rays, {(dataset.labels == 0).sum()} fractured, "
      f"{sum(len(b) for b in dataset.boxes)} fracture boxes drawn by the radiologists")

### Are there copies of the same X-ray?

Some X-rays may be saved more than once (resized, flipped, brighter). If one copy is used for training and
another for testing, the test measures memory, not generalisation. We look for **near-duplicates**: images
whose 32x32 thumbnails have a cosine similarity >= `DUP_THRESHOLD` for one of the 8 flips / 90-degree
rotations (copies rotated by other angles are missed, so these counts are a lower bound).

In [ ]:
dup = data.study_duplicates(dataset, DUP_THRESHOLD, device=device)
plots.show_duplicate_pairs(dataset, dup["pairs"], title="Near-duplicates: the same X-ray saved twice",
                           save_to=FIG_DIR / "duplicates.png")
dup["table"]

### Train / val / test split

FracAtlas has no official split. We make a stratified split of about 70 / 15 / 15 % in which all the copies
of one X-ray stay in the same part: **train** (cross-validation and final training), **val** (only to pick the
best epoch of the final training) and **test** (used once, at the end).

In [ ]:
split = data.split_indices(dataset.labels, dup["groups"], seed=SEED)
sets = {name: dataset.subset(idx) for name, idx in split.items()}
train_groups = dup["groups"][split["train"]]   # the cross-validation folds are grouped too
data.summary_table(sets)

In [ ]:
plots.plot_dataset_overview(sets, save_to=FIG_DIR / "dataset_overview.png")
plots.show_samples(sets["train"], title="Training X-rays (dashed: the radiologists' fracture boxes)",
                   save_to=FIG_DIR / "samples_train.png")

## 2. Models

| model | feature extractor (what differs) | classifier (identical) |
|---|---|---|
| `cnn` | 4 learned conv blocks (conv, BN, ReLU, 2x2 max-pool): 32 filters 7x7, then 64, 128, 256 of 3x3 -> 256 x 14 x 14 | flatten -> 512 -> 128 -> 2 |
| `scatnet` | fixed scattering transform (Kymatio, J=4 scales, L=8 orientations, 2nd order) -> log -> BN -> 417 x 14 x 14 | flatten -> 512 -> 128 -> 2 |
| `resnet18` | ImageNet-pretrained ResNet18, fine-tuned (reference) -> 512 | flatten -> 512 -> 128 -> 2 |

CNN and ScatNet end on the same 14x14 grid (224 / 2^4), so their classifiers differ **only in the
number of input neurons**, as the exam requires (checked below). ScatNet learns no filters: its only
trainable parts are the batch-norm of the coefficients and the classifier.

In [ ]:
layouts = {name: models.classifier_layout(models.build_model(name, pretrained=False)) for name in MODELS}
assert all(layout == layouts["cnn"] for layout in layouts.values()), "the classifiers differ"
print("identical classifier:", layouts["cnn"])
models.summary(MODELS)

## 3. k-fold cross-validation on the training set

Stratified, **group-aware** k-fold CV (copies of an X-ray never cross folds). Each fold trains a fresh
model for `EPOCHS` epochs (AdamW, weight decay 1e-4, cosine learning-rate schedule, augmentation: flips,
rotation +-10 degrees, shift, zoom, brightness/contrast). The cross-entropy weights each class by its
inverse frequency, otherwise the few fractured X-rays would count little and be missed. The fold score is
taken at the **last epoch**: nothing is selected on the validation fold, so the estimate is not optimistic.
F1 is computed for the **fractured** class.

In [ ]:
cv = {}
for name in MODELS:
    if TRAIN:
        cv[name] = training.cross_validate(name, sets["train"], train_groups, device, K_FOLDS, EPOCHS, BATCH_SIZE,
                                           LR[name], seed=SEED, num_workers=NUM_WORKERS, results_dir=RESULTS_DIR)
    else:
        cv[name] = training.load_json(RESULTS_DIR / "cv" / f"{name}.json")
training.cv_table(cv)   # mean ± std over the folds, in %

## 4. Final models and test set

Each model is retrained on the **whole training split**; the val split picks its best epoch. Then the
test split is used once. Learning curves show train and validation on the same axes.

In [ ]:
trained, final = {}, {}
for name in MODELS:
    if TRAIN:
        trained[name], final[name] = training.train_final(name, sets["train"], sets["val"], device, EPOCHS, BATCH_SIZE,
                                                          LR[name], seed=SEED, num_workers=NUM_WORKERS,
                                                          results_dir=RESULTS_DIR)
    else:
        trained[name] = training.load_model(name, RESULTS_DIR, device)
        final[name] = training.load_json(RESULTS_DIR / "final" / f"{name}.json")
plots.plot_learning_curves(cv, final, save_to=FIG_DIR / "learning_curves.png")

In [ ]:
test = {name: training.evaluate_test(trained[name], sets["test"], device) for name in MODELS}
training.test_table(test)   # accuracy [95% bootstrap interval], other metrics in %

In [ ]:
plots.plot_confusion_matrices(test, class_names, save_to=FIG_DIR / "confusion_matrices.png")
plots.plot_roc(test, save_to=FIG_DIR / "roc.png")

**Is the difference between two models real?** McNemar's exact test compares two models on the same
test images, using only the images where exactly one of them is right.

In [ ]:
mcnemar = training.mcnemar_table(test)
mcnemar

## 5. Model comparison and choice of the best model

The best model is the one with the highest **cross-validation** score (`SELECT_BY`), computed on the
training split only; the test split then confirms (or not) the choice. Choosing on the test split would
make the test score of the chosen model optimistic.

In [ ]:
comparison = training.comparison_table(cv, test, final)
BEST = training.select_best(cv, SELECT_BY)
plots.plot_model_comparison(comparison, best=BEST, save_to=FIG_DIR / "model_comparison.png")
comparison

In [ ]:
errors = training.error_table(test, class_names)   # a missed fracture is the costly error
print(errors.attrs["wrong for every model"], "test images are misclassified by every model")
errors

## 6. Filters: learned (CNN) vs fixed (ScatNet)

The CNN **learns** its 32 first-layer filters from the X-rays; ScatNet uses **fixed** Morlet wavelets,
one per scale and orientation. The frequency plot shows which spatial frequencies and orientations each
first layer covers: the wavelets tile the plane by design, the CNN concentrates where the data needs it.

In [ ]:
plots.plot_conv_filters(trained["cnn"].first_conv.weight, "CNN: the 32 learned 7x7 filters of the first layer",
                        save_to=FIG_DIR / "filters_cnn.png")
plots.plot_scattering_filters(trained["scatnet"].J, trained["scatnet"].L, save_to=FIG_DIR / "filters_scatnet.png")
plots.plot_frequency_coverage(trained["cnn"].first_conv.weight, trained["scatnet"].J, trained["scatnet"].L,
                              save_to=FIG_DIR / "frequency_coverage.png")

In [ ]:
if "resnet18" in trained:
    plots.plot_conv_filters(trained["resnet18"].first_conv.weight, "ResNet18: first 32 of its 64 7x7 filters (RGB summed)",
                            save_to=FIG_DIR / "filters_resnet18.png")

## 7. Explaining the models: six XAI methods

| method | idea | ScatNet? |
|---|---|---|
| Saliency | \|d score / d pixel\| | yes |
| Integrated Gradients | gradients along the path black image -> image | yes |
| Guided Backprop | only positive signals go back through the ReLUs | only the classifier's ReLUs |
| Grad-CAM | last conv maps weighted by their gradient | **no**: no learned conv layer |
| Occlusion | score drop when a 16x16 window turns black | yes |
| LIME | linear model on random on/off 16x16 patches | yes |

We explain `XAI_PER_CLASS` test images per class that every model classifies correctly (figures: 2 per class).
Red = evidence for the class, blue = against.

In [ ]:
XAI_MODELS = list(dict.fromkeys(["cnn", "scatnet", BEST]))  # the two models the exam requires + the best one
correct = np.all([np.array(test[n]["y_pred"]) == sets["test"].labels for n in XAI_MODELS], axis=0)
xai_idx = data.pick_images(sets["test"].labels, XAI_PER_CLASS, seed=SEED, allowed=correct)
xai_images, xai_labels = sets["test"].tensors(xai_idx)
n_frac = int((xai_labels == 0).sum())
SHOW = [0, 1, n_frac, n_frac + 1]                          # 2 fractured + 2 not fractured
xai.applicability({name: trained[name] for name in XAI_MODELS})

In [ ]:
attributions, seconds = {}, {}
for name in XAI_MODELS:
    print(name)
    attributions[name], seconds[name] = xai.explain_all(
        trained[name], xai_images, xai_labels, methods={**xai.METHODS, **xai.SCRATCH},
        cache=RESULTS_DIR / "attributions" / f"{name}.npz", recompute=TRAIN, **XAI_PARAMS)

In [ ]:
for name in XAI_MODELS:
    plots.plot_attribution_grid(xai_images, xai_labels, attributions[name], class_names, methods=xai.METHODS,
                                indices=SHOW, title=f"{plots.MODEL_LABELS[name]}: six XAI methods",
                                save_to=FIG_DIR / f"xai_{name}.png")

## 8. Our Occlusion vs Captum's Occlusion

`src/occlusion_scratch.py` implements Occlusion with plain PyTorch: slide a 16x16 black window
(stride 8), measure the drop of the class score, average the drops over the windows covering each
pixel. Same window grid and averaging as Captum, so the two maps should match up to float error.

In [ ]:
xai.scratch_vs_captum(attributions)

In [ ]:
plots.plot_scratch_vs_captum(xai_images, attributions, index=SHOW[0],
                             save_to=FIG_DIR / "occlusion_scratch_vs_captum.png")

## 9. Which method gives the best attributions?

Looking is not enough, so we measure two things on all the explained images:
- **faithfulness (deletion test)**: black out 16x16 patches from the most to the least important
  according to the map and follow the probability of the class. A map that points at what the model
  really uses makes it drop fast: **lower area = better**. *Random* order is the reference to beat.
- **agreement**: Spearman correlation between the patch-level maps of two methods.

In [ ]:
curves = {name: xai.deletion_curves(trained[name], xai_images, xai_labels, attributions[name]) for name in XAI_MODELS}
deletion = {name: xai.deletion_table(curves[name]) for name in XAI_MODELS}
plots.plot_deletion_curves(curves, save_to=FIG_DIR / "deletion_curves.png")
pd.DataFrame({plots.MODEL_LABELS[n]: d["deletion AUC (mean)"] for n, d in deletion.items()})   # lower = better

In [ ]:
plots.plot_agreement({name: xai.agreement_matrix(attributions[name]) for name in XAI_MODELS},
                     save_to=FIG_DIR / "xai_agreement.png")
pd.DataFrame(seconds).round(3).rename(columns=plots.MODEL_LABELS)   # computation time, seconds per image

## 10. Where is the fracture? From heatmap to box

Each heatmap becomes a box (`src/localize.py`): keep the evidence for *fractured*, keep the pixels above
`BOX_THRESHOLD` x the strongest, box around the blob with the most evidence. On every fractured test X-ray:
- **hit rate**: the hottest point is inside a radiologist's box (pointing game); *Random* is the reference;
- **IoU**: overlap of our box with the radiologist's box.

In the figures the evidence outside our box stays faint, so a model that looks at a label or a plate is not hidden.

In [ ]:
frac = np.flatnonzero(sets["test"].labels == 0)            # every fractured test X-ray (each has boxes)
box_images, _ = sets["test"].tensors(frac)
true_boxes = [sets["test"].boxes[i] for i in frac]
box_maps = {name: xai.explain_all(trained[name], box_images, np.zeros(len(frac), dtype=int),  # explain 'fractured'
                                  cache=RESULTS_DIR / "attributions" / f"{name}_boxes.npz", recompute=TRAIN,
                                  **XAI_PARAMS)[0] for name in XAI_MODELS}
localization = {name: localize.localization_table(box_maps[name], true_boxes, BOX_THRESHOLD) for name in XAI_MODELS}

In [ ]:
for name in XAI_MODELS:
    plots.plot_fracture_boxes(box_images[:3], box_maps[name], true_boxes[:3], methods=xai.METHODS,
                              threshold=BOX_THRESHOLD, title=f"{plots.MODEL_LABELS[name]}: where is the fracture?",
                              save_to=FIG_DIR / f"boxes_{name}.png")

In [ ]:
pd.concat({plots.MODEL_LABELS[n]: t for n, t in localization.items()}, axis=1)   # hit rate and IoU: higher = better

## 11. Full pipeline: classifier + YOLO + explanation (after Linda, 2025)

1. **YOLOv8** learns the radiologists' boxes of our training split (FracAtlas paper baseline: mAP@0.5 = 0.562);
2. the **best classifier** (section 5) says fractured or not;
3. the **best XAI method** (most hits among the methods that beat Random) explains it: inside YOLO's box?
4. **verdict**: *fracture* / *no fracture* when classifier and YOLO agree, otherwise *needs review*.

From the paper we leave out the CT scans (FracAtlas has X-rays only) and the graph network.

In [ ]:
yolo_data = detect.write_yolo_dataset(sets, CACHE_DIR / "yolo")   # fractured X-rays + boxes, same split
detector = (detect.train(yolo_data, RESULTS_DIR / "detector", seed=SEED, **YOLO_PARAMS) if TRAIN
            else detect.load(RESULTS_DIR / "detector"))
detector_scores = detect.box_metrics(detector, yolo_data, imgsz=YOLO_PARAMS["imgsz"])   # fractured test X-rays
detector_scores

In [ ]:
detections = detect.find_boxes(detector, sets["test"].paths, data.IMAGE_SIZE, conf=YOLO_CONF,
                               imgsz=YOLO_PARAMS["imgsz"])               # boxes on every test X-ray
detector_boxes = localize.score_boxes([detections[i] for i in frac], true_boxes, box_images.shape[-2:])
plots.plot_localization(localization, detector=detector_boxes["hit rate (%)"], save_to=FIG_DIR / "localization.png")
localize.detector_vs_xai(localization, detector_boxes, plots.MODEL_LABELS)   # who finds the fracture?

In [ ]:
BOX_METHOD = localize.choose_method(localization[BEST], deletion[BEST])
print(f"classifier: {plots.MODEL_LABELS[BEST]} | explanation: {BOX_METHOD}")
cases = pipeline.run(trained[BEST], sets["test"], test[BEST]["prob_positive"], detections, BOX_METHOD, device,
                     **XAI_PARAMS)
pipeline.outcomes(cases)   # rows: the truth, columns: the verdict

In [ ]:
plots.plot_pipeline_examples(sets["test"], cases, save_to=FIG_DIR / "pipeline_examples.png")
print("\n\n".join(cases["report"].head(3)))
pipeline.scores(cases)

## 12. Export the results

`results/summary.json` holds every number; `results/latex/` holds the macros and tables used by the
presentation (`presentation/`, rebuilt with `make -C presentation`).

In [ ]:
settings = dict(models=MODELS, k_folds=K_FOLDS, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED,
                dup_threshold=DUP_THRESHOLD, xai_per_class=XAI_PER_CLASS, xai_params=XAI_PARAMS,
                box_threshold=BOX_THRESHOLD, yolo_params=YOLO_PARAMS,
                yolo_conf=YOLO_CONF, quick=QUICK)
summary = report.export(RESULTS_DIR, settings, data.summary_table(sets), dup["table"], cv, final, test, BEST,
                        SELECT_BY, mcnemar, deletion, seconds, xai.scratch_vs_captum(attributions), comparison,
                        localization, dict(detector=detector_scores, detector_boxes=detector_boxes,
                                           cases=pipeline.scores(cases), box_method=BOX_METHOD))
report.key_findings(summary);

## 13. Discussion

The numbers are printed above (`key_findings`) and in each section.

**Which model and why.** ScatNet learns only a classifier on fixed wavelets: stable, but generic, so it cannot
adapt to thin, irregular fracture lines. The CNN learns filters for X-rays; ResNet18 adds ImageNet pre-training.
The model is chosen on cross-validation, confirmed on the test set; McNemar says if a gap is real.

**Data.** Copies of an X-ray stay in one split and one fold. Only 1 X-ray in 6 is fractured: "always not
fractured" already scores ~82% accuracy, so the loss is weighted and F1, recall and AUC matter more.

**Filters.** The CNN's 7x7 filters become oriented edge and blob detectors, like the Morlet wavelets but not
spread evenly over scales and orientations; the wavelets cover all of them, needed or not.

**Which XAI method.** Gradients are sharp but noisy; Integrated Gradients satisfies completeness; Occlusion and
LIME test the model directly (faithful but slow, patch resolution); Grad-CAM is coarse. The deletion test and
the hit rate rank them on each model.

**When a method cannot be used.** Grad-CAM needs a *learned* conv feature map: ScatNet has none. Guided
Backprop acts on ReLUs, but ScatNet's non-linearity is a modulus, so only its classifier is "guided".

**Our Occlusion** equals Captum's up to floating-point error (section 8).

**Boxes and pipeline.** YOLO is trained *with* the boxes, the classifiers never see one: YOLO's hit rate is the
level an explanation could reach, Random the level of a guess. The pipeline decides when classifier and YOLO
agree and asks for review otherwise; the share of fractures ending in *no fracture* matters most.

**Limitations.** Copies rotated by arbitrary angles are missed; FracAtlas is small and unbalanced (about a
hundred fractured test X-rays); one baseline (black), one patch size and one threshold.